## Import libraries

In [1]:
# import os
# from pathlib import Path

import pandas as pd
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
from psycopg2.extras import execute_batch

## Functions to connect to db, create new db.
## Created new db 'sp500_rpoject'

In [2]:
DB_NAME = "sp500_project"
USER = "postgres"
HOST = "localhost"
PORT = "5432"
PWD_FILE = "pwd.txt"

with open(PWD_FILE, "r") as f:
    PWD = f.read().strip()

In [3]:
def connect(db_nm: str = 'postgres',
            pwd: str = PWD,
            usr_nm: str = USER, 
            host: str = HOST,
            port: str = PORT,
           ) -> psycopg2.extensions.connection | None:
    conn = None
    
    try:
            conn = psycopg2.connect(
            dbname=db_nm,
            user=usr_nm,
            password=pwd,
            host=host,
            port=port
            )
    except Exception as e:
        print(f'Failed to connect to database. Error: {str(e)}')
    else:
        print(f'Connected to database {db_nm}')
    finally:
        return conn

def create_database(db_name: str,
                    usr_nm: str = USER,
                    pwd: str = PWD,
                    host: str = HOST,
                    port: str = PORT) -> None:
    conn = psycopg2.connect(
        dbname='postgres',
        user=usr_nm,
        password=pwd,
        host=host,
        port=port
    )
    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
    cur = conn.cursor()

    try:
        cur.execute(f"CREATE DATABASE {db_name};")
    except Exception as e:
        print(f"Failed to create database '{db_name}'. Error: {str(e)}")
    else:
        print(f"Database '{db_name}' created.")
    finally:
        cur.close()
        conn.close()

In [4]:
create_database(DB_NAME)

conn = connect(DB_NAME)
cur = conn.cursor()

cur.close()
conn.close()

Failed to create database 'sp500_project'. Error: database "sp500_project" already exists

Connected to database sp500_project


## Clean data before converting to SQL. Problem: null values for adj_close in stocks

In [36]:
stocks_clean = stocks.copy()
companies_clean = companies.copy()
index_clean = index_df.copy()

stocks_clean = stocks_clean.rename(columns={
    'Date': 'date',
    'Symbol': 'symbol',
    'Adj Close': 'adj_close',
    'Close': 'close',
    'High': 'high',
    'Low': 'low',
    'Open': 'open',
    'Volume': 'volume'
})

companies_clean = companies_clean.rename(columns={
    'Exchange': 'exchange',
    'Symbol': 'symbol',
    'Shortname': 'shortname',
    'Longname': 'longname',
    'Sector': 'sector',
    'Industry': 'industry',
    'Currentprice': 'current_price',
    'Marketcap': 'market_cap',
    'Ebitda': 'ebitda',
    'Revenuegrowth': 'revenue_growth',
    'City': 'city',
    'State': 'state',
    'Country': 'country',
    'Fulltimeemployees': 'fulltime_employees',
    'Longbusinesssummary': 'long_business_summary',
    'Weight': 'weight'
})

index_clean = index_clean.rename(columns={
    'Date': 'date',
    'S&P500': 'sp500'
})

stocks_clean['date'] = pd.to_datetime(stocks_clean['date'])
index_clean['date'] = pd.to_datetime(index_clean['date'])

stocks_clean = stocks_clean.dropna(subset=['adj_close']).copy()

stocks_clean = stocks_clean.sort_values(['symbol', 'date']).reset_index(drop=True)
companies_clean = companies_clean.sort_values('symbol').reset_index(drop=True)
index_clean = index_clean.sort_values('date').reset_index(drop=True)

stocks_clean['date'] = stocks_clean['date'].dt.date
index_clean['date'] = index_clean['date'].dt.date

## Converting to SQL

In [47]:
conn = connect(DB_NAME)
cur = conn.cursor()

cur.execute("""
DROP TABLE IF EXISTS stocks;
DROP TABLE IF EXISTS companies;
DROP TABLE IF EXISTS market_index;
""")

cur.execute("""
CREATE TABLE companies (
    exchange TEXT,
    symbol TEXT PRIMARY KEY,
    shortname TEXT,
    longname TEXT,
    sector TEXT,
    industry TEXT,
    current_price DOUBLE PRECISION,
    market_cap BIGINT,
    ebitda DOUBLE PRECISION,
    revenue_growth DOUBLE PRECISION,
    city TEXT,
    state TEXT,
    country TEXT,
    fulltime_employees DOUBLE PRECISION,
    long_business_summary TEXT,
    weight DOUBLE PRECISION
);
""")

cur.execute("""
CREATE TABLE market_index (
    date DATE PRIMARY KEY,
    sp500 DOUBLE PRECISION
);
""")

cur.execute("""
CREATE TABLE stocks (
    date DATE,
    symbol TEXT,
    adj_close DOUBLE PRECISION,
    close DOUBLE PRECISION,
    high DOUBLE PRECISION,
    low DOUBLE PRECISION,
    open DOUBLE PRECISION,
    volume DOUBLE PRECISION,
    PRIMARY KEY (symbol, date)
);
""")

print("Tables created.")


# Inserting values into tables
companies_rows = list(companies_clean.itertuples(index=False, name=None))

insert_companies = """
INSERT INTO companies (
    exchange, symbol, shortname, longname, sector, industry,
    current_price, market_cap, ebitda, revenue_growth,
    city, state, country, fulltime_employees,
    long_business_summary, weight
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
"""

execute_batch(cur, insert_companies, companies_rows, page_size=1000)
print("companies loaded.")

index_rows = list(index_clean.itertuples(index=False, name=None))

insert_index = """
INSERT INTO market_index (date, sp500)
VALUES (%s, %s);
"""

execute_batch(cur, insert_index, index_rows, page_size=1000)
conn.commit()
cur.close()
conn.close()

print("market_index loaded.")

def load_stocks_in_chunks(df, chunk_size=50000):
    conn = connect(DB_NAME)
    cur = conn.cursor()

    insert_stocks = """
    INSERT INTO stocks (
        date, symbol, adj_close, close, high, low, open, volume
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s);
    """

    total_rows = len(df)

    for start in range(0, total_rows, chunk_size):
        end = min(start + chunk_size, total_rows)
        chunk = df.iloc[start:end]

        rows = list(chunk.itertuples(index=False, name=None))
        execute_batch(cur, insert_stocks, rows, page_size=5000)
        conn.commit()

        print(f"Loaded rows {start} to {end} out of {total_rows}")

    cur.close()
    conn.close()
    print("stocks loaded.")

load_stocks_in_chunks(stocks_clean)

Connected to database sp500_project
Tables created.
companies loaded.
market_index loaded.
Connected to database sp500_project
Loaded rows 0 to 50000 out of 617831
Loaded rows 50000 to 100000 out of 617831
Loaded rows 100000 to 150000 out of 617831
Loaded rows 150000 to 200000 out of 617831
Loaded rows 200000 to 250000 out of 617831
Loaded rows 250000 to 300000 out of 617831
Loaded rows 300000 to 350000 out of 617831
Loaded rows 350000 to 400000 out of 617831
Loaded rows 400000 to 450000 out of 617831
Loaded rows 450000 to 500000 out of 617831
Loaded rows 500000 to 550000 out of 617831
Loaded rows 550000 to 600000 out of 617831
Loaded rows 600000 to 617831 out of 617831
stocks loaded.


In [48]:
conn = connect(DB_NAME)
cur = conn.cursor()

cur.execute('''
    SELECT * FROM stocks LIMIT 5
''')
print(cur.fetchall())

cur.close()
conn.close()

Connected to database sp500_project
[(datetime.date(2013, 1, 2), 'ABBV', 21.629180908203125, 35.119998931884766, 35.400001525878906, 34.099998474121094, 34.91999816894531, 13767900.0), (datetime.date(2013, 1, 3), 'ABBV', 21.450580596923828, 34.83000183105469, 35.0, 34.15999984741211, 35.0, 16739300.0), (datetime.date(2013, 1, 4), 'ABBV', 21.179594039916992, 34.38999938964844, 34.88999938964844, 34.25, 34.619998931884766, 21372100.0), (datetime.date(2013, 1, 7), 'ABBV', 21.222700119018555, 34.459999084472656, 35.45000076293945, 34.150001525878906, 34.150001525878906, 17897100.0), (datetime.date(2013, 1, 8), 'ABBV', 20.76080894470215, 33.709999084472656, 34.63999938964844, 33.36000061035156, 34.290000915527344, 17863300.0)]


In [49]:
conn = connect(DB_NAME)
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM stocks;")
print("stocks rows:", cur.fetchone()[0])

cur.execute("SELECT COUNT(*) FROM companies;")
print("companies rows:", cur.fetchone()[0])

cur.execute("SELECT COUNT(*) FROM market_index;")
print("market_index rows:", cur.fetchone()[0])

cur.close()
conn.close()

Connected to database sp500_project
stocks rows: 617831
companies rows: 502
market_index rows: 2517


## Step 1. Joins

In [5]:
def show_res_of_query(qr: str, cr: psycopg2.extensions.cursor) -> pd.DataFrame:
    cr.execute(qr)
    rows = cr.fetchall()
    cols = [desc[0] for desc in cr.description]
    return pd.DataFrame(rows, columns=cols)

In [87]:
conn = connect(DB_NAME)
cur = conn.cursor()

query = """
DROP VIEW IF EXISTS step1_ans;

CREATE VIEW step1_ans AS
SELECT
    s.*,
    c.sector,
    c.industry,
    c.weight,
    m.sp500
FROM stocks s
LEFT JOIN companies c
    ON s.symbol = c.symbol
LEFT JOIN market_index m
    ON s.date = m.date;
"""
cur.execute(query)
conn.commit()

query = '''
SELECT * FROM step1_ans LIMIT 5
'''
print(show_res_of_query(query, cur))

cur.close()
conn.close()

print("View step1_ans was created. \n It is the result of step1 of the project")

Connected to database sp500_project
         date symbol  adj_close      close       high        low       open  \
0  2013-01-02   ABBV  21.629181  35.119999  35.400002  34.099998  34.919998   
1  2013-01-03   ABBV  21.450581  34.830002  35.000000  34.160000  35.000000   
2  2013-01-04   ABBV  21.179594  34.389999  34.889999  34.250000  34.619999   
3  2013-01-07   ABBV  21.222700  34.459999  35.450001  34.150002  34.150002   
4  2013-01-08   ABBV  20.760809  33.709999  34.639999  33.360001  34.290001   

       volume      sector                      industry    weight sp500  
0  13767900.0  Healthcare  Drug Manufacturers - General  0.005582  None  
1  16739300.0  Healthcare  Drug Manufacturers - General  0.005582  None  
2  21372100.0  Healthcare  Drug Manufacturers - General  0.005582  None  
3  17897100.0  Healthcare  Drug Manufacturers - General  0.005582  None  
4  17863300.0  Healthcare  Drug Manufacturers - General  0.005582  None  
View step1_ans was created. 
 It is the resul

In [90]:
conn = connect(DB_NAME)
cur = conn.cursor()

query = '''
DROP TABLE IF EXISTS step2_start;
    
CREATE TABLE step2_start AS
SELECT * FROM step1_ans
WHERE sp500 IS NOT NULL;
'''
cur.execute(query)
conn.commit()

query = '''
SELECT * FROM step2_start LIMIT 5
'''
print(show_res_of_query(query, cur))

cur.close()
conn.close()

Connected to database sp500_project
         date symbol  adj_close      close       high        low       open  \
0  2014-12-22   ABBV  44.277233  66.970001  68.250000  66.830002  68.040001   
1  2014-12-23   ABBV  42.545006  64.349998  67.320000  64.019997  67.230003   
2  2014-12-24   ABBV  43.774754  66.209999  66.949997  64.750000  64.750000   
3  2014-12-26   ABBV  44.283840  66.980003  67.239998  66.510002  66.510002   
4  2014-12-29   ABBV  44.389626  67.139999  67.370003  66.419998  66.580002   

       volume      sector                      industry    weight    sp500  
0  12876600.0  Healthcare  Drug Manufacturers - General  0.005582  2078.54  
1  12123000.0  Healthcare  Drug Manufacturers - General  0.005582  2082.17  
2   4705500.0  Healthcare  Drug Manufacturers - General  0.005582  2081.88  
3   4158200.0  Healthcare  Drug Manufacturers - General  0.005582  2088.77  
4   3872800.0  Healthcare  Drug Manufacturers - General  0.005582  2090.57  


## Step 2. Returns

In [91]:
conn = connect(DB_NAME)
cur = conn.cursor()

query = '''
DROP TABLE IF EXISTS step2_returns;

CREATE TABLE step2_returns AS
SELECT 
    final.*,
    stock_return - market_return AS market_adj_return
FROM (
    SELECT 
        s.date,
        symbol,
        adj_close,
        adj_close / LAG(adj_close) OVER (PARTITION BY symbol ORDER BY s.date ASC) - 1 AS stock_return,
        market_return,
        s.sector,
        s.industry,
        s.weight
    FROM step2_start s
    LEFT JOIN (
        SELECT m.date, m.sp500,
        sp500 / LAG(sp500) OVER () - 1 AS market_return
        FROM market_index m
    ) m ON s.date = m.date
    ORDER BY symbol, s.date
) AS final
WHERE stock_return IS NOT NULL AND market_return IS NOT NULL;
'''
cur.execute(query)
conn.commit()

query = '''
SELECT * FROM step2_returns LIMIT 5
'''
print(show_res_of_query(query, cur))

cur.close()
conn.close()

Connected to database sp500_project
         date symbol  adj_close  stock_return  market_return      sector  \
0  2014-12-23   ABBV  42.545006     -0.039122       0.001746  Healthcare   
1  2014-12-24   ABBV  43.774754      0.028905      -0.000139  Healthcare   
2  2014-12-26   ABBV  44.283840      0.011630       0.003310  Healthcare   
3  2014-12-29   ABBV  44.389626      0.002389       0.000862  Healthcare   
4  2014-12-30   ABBV  43.834248     -0.012511      -0.004889  Healthcare   

                       industry    weight  market_adj_return  
0  Drug Manufacturers - General  0.005582          -0.040869  
1  Drug Manufacturers - General  0.005582           0.029044  
2  Drug Manufacturers - General  0.005582           0.008320  
3  Drug Manufacturers - General  0.005582           0.001527  
4  Drug Manufacturers - General  0.005582          -0.007623  


In [96]:
conn = connect(DB_NAME)
cur = conn.cursor()

query = '''
DROP TABLE IF EXISTS step2_ans;

CREATE TABLE step2_ans AS
SELECT 
    final.date,
    final.symbol,
    final.adj_close,
    final.sector,
    final.industry,
    final.weight,
    final.stock_return,
    final.market_return,
    final.market_adj_return,
    corr_60 * std_stock_60 / std_market_60 AS beta_60
FROM(
    SELECT 
        s_ret.*,
        CORR(stock_return, market_return) OVER (
            PARTITION BY symbol ORDER BY date ASC
            ROWS BETWEEN 59 PRECEDING AND CURRENT ROW
                                                ) AS corr_60,
        STDDEV_SAMP(stock_return) OVER (
            PARTITION BY symbol ORDER BY date ASC
            ROWS BETWEEN 59 PRECEDING AND CURRENT ROW
                                                ) AS std_stock_60,
        STDDEV_SAMP(market_return) OVER (
            PARTITION BY symbol ORDER BY date ASC
            ROWS BETWEEN 59 PRECEDING AND CURRENT ROW
                                                ) AS std_market_60,
        COUNT(*) OVER (
            PARTITION BY symbol ORDER BY date
            ROWS BETWEEN 59 PRECEDING AND CURRENT ROW
                                                ) AS n_obs_60
    FROM step2_returns s_ret
    ) final
WHERE corr_60 IS NOT NULL AND std_stock_60 IS NOT NULL AND std_market_60 IS NOT NULL AND n_obs_60 = 60
;
'''
cur.execute(query)
conn.commit()

query = '''
SELECT * FROM step2_ans LIMIT 5
'''
print(show_res_of_query(query, cur))

cur.close()
conn.close()

print('\n Table step2_ans was created. It is the result of step2 of the project')

Connected to database sp500_project
         date symbol  adj_close      sector                      industry  \
0  2015-03-20   ABBV  40.233265  Healthcare  Drug Manufacturers - General   
1  2015-03-23   ABBV  40.286549  Healthcare  Drug Manufacturers - General   
2  2015-03-24   ABBV  39.720348  Healthcare  Drug Manufacturers - General   
3  2015-03-25   ABBV  38.781143  Healthcare  Drug Manufacturers - General   
4  2015-03-26   ABBV  38.148315  Healthcare  Drug Manufacturers - General   

     weight  stock_return  market_return  market_adj_return   beta_60  
0  0.005582     -0.012911       0.009013          -0.021923  1.166020  
1  0.005582      0.001324      -0.001746           0.003070  1.177493  
2  0.005582     -0.014054      -0.006139          -0.007915  1.187462  
3  0.005582     -0.023645      -0.014559          -0.009087  1.195398  
4  0.005582     -0.016318      -0.002377          -0.013940  1.200035  

 Table step2_ans was created. It is the result of step2 of the proje